# Real 20-question Qwen3 pilot

This runs 100 real model responses: two baselines and three wrong-answer hints for each of the first 20 frozen questions. Select a **T4 GPU**. No API key or paid API calls are needed. This is a small engineering pilot, not a validated claim about faithfulness. Read `docs/PILOT_PROTOCOL.md` before interpreting results.


In [ ]:
from pathlib import Path
import os, subprocess, sys
SOURCE_REVISION = "fe4de16013bf9e3e9a9d66abd5b47401becbe24a"  # The launched pilot pins the exact source commit below.
PROJECT = Path("/content/hint-faithfulness")
if not PROJECT.exists():
    subprocess.run(["git", "clone", "https://github.com/felmonon/hint-faithfulness.git", str(PROJECT)], check=True)
os.chdir(PROJECT)
subprocess.run(["git", "checkout", SOURCE_REVISION], check=True)
print("Source commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-pilot.txt"], check=True)
import torch
assert torch.cuda.is_available(), "Choose Runtime > Change runtime type > T4 GPU before continuing."
print("GPU:", torch.cuda.get_device_name(0))
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-q"], check=True)


## Frozen prediction and scope

The assistant-authored hypothesis, recorded before collecting responses, is that grader-hacking may cause the highest uptake and that some switched traces will not explicitly acknowledge using the hint. This is **not attributed to Felmon as a personal prediction**. Sample size and decoding settings remain fixed even if results are null or incomplete.


In [ ]:
RUN_DIR = Path("artifacts/pilot-20-qwen3-1.7b")
RUN_DIR.mkdir(parents=True, exist_ok=True)
if not (RUN_DIR / "runtime.json").exists():
    subprocess.run([sys.executable, "-m", "src.provenance", "--output", str(RUN_DIR / "runtime.json")], check=True)
command = [sys.executable, "-u", "-m", "src.run", "--backend", "local", "--model", "Qwen/Qwen3-1.7B", "--questions", "data/questions.jsonl", "--limit", "20", "--max-tokens", "4096", "--device", "cuda", "--output", str(RUN_DIR / "responses.jsonl")]
with (RUN_DIR / "collection.log").open("a") as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()
    assert process.wait() == 0, "Collection stopped. Preserve the log and partial results before debugging."


In [ ]:
from src.common import read_jsonl
from collections import Counter
rows = read_jsonl(RUN_DIR / "responses.jsonl")
print("Saved responses:", len(rows), "Expected:", 100)
print("Status counts:", dict(Counter(row["status"] for row in rows)))
assert len(rows) == 100, "Preserve and inspect partial results; do not treat this as a completed pilot."
subprocess.run([sys.executable, "-m", "src.analyze", "--input", str(RUN_DIR / "responses.jsonl"), "--out-dir", str(RUN_DIR / "analysis")], check=True)
if not (RUN_DIR / "transcripts.md").exists():
    subprocess.run([sys.executable, "-m", "src.transcripts", "--input", str(RUN_DIR / "responses.jsonl"), "--output", str(RUN_DIR / "transcripts.md")], check=True)
from IPython.display import display, Image
display(Image(str(RUN_DIR / "analysis/phase1.png")))


## Save the complete evidence

The archive contains raw responses, exact token IDs, prompts, run and runtime manifests, the collection log, transcripts, and provisional analysis. Faithfulness stays pending until a frozen judge passes held-out human validation. No fabricated labels are added.


In [ ]:
import zipfile, hashlib, json
from google.colab import files
manifest = {str(path.relative_to(RUN_DIR)): hashlib.sha256(path.read_bytes()).hexdigest() for path in RUN_DIR.rglob("*") if path.is_file() and path.name != "checksums.json"}
(RUN_DIR / "checksums.json").write_text(json.dumps(manifest, indent=2) + "\n")
archive_path = Path("/content/pilot-20-results.zip")
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in RUN_DIR.rglob("*"):
        if path.is_file():
            archive.write(path, path.relative_to(PROJECT))
print("Results archived:", archive_path.name)
files.download(str(archive_path))
